# 四只股票金融数据分析大作业

**标的：** GOOGL（谷歌）、AVGO（博通）、SLV（白银 ETF）、NVDA（英伟达）  
**数据：** 近两年日线收盘价  
**工具：** Pandas、NumPy、yfinance、matplotlib

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "src" / "config.py").exists():
        sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Cannot find project root (src/config.py)")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TICKERS, IMAGES_DIR
from src.data_fetch import run_pipeline, load_raw, build_close_wide
from src.analysis import run_full_analysis, load_prices_wide
from src.plotting import generate_all_figures
from src.utils import summary_stats, vectorized_summary, log_returns_from_prices

sns.set_theme(style="whitegrid")
print("Project root:", PROJECT_ROOT)
print("Tickers:", TICKERS)

## 第一步：数据采集与清洗

使用 `yfinance` 批量下载数据，保存 CSV，再合并为收盘价宽表。

In [ ]:
# 若 yfinance 限流，改为 run_pipeline(demo=True)
USE_DEMO = True  # 有网络且未限流时改为 False
prices = run_pipeline(demo=USE_DEMO)
prices.head()

In [ ]:
sample = load_raw("GOOGL")
print("--- head ---")
display(sample.head())
print("--- info ---")
sample.info()
print("--- describe ---")
display(sample.describe())

## 第二步：收益率与 NumPy 向量化

简单收益率 `pct_change()`，对数收益率 `np.log()`，统计量用 `np.mean/std/var`，累计收益用 `np.cumsum()`。

In [ ]:
results = run_full_analysis(prices)
rets = results["simple_returns"]
log_rets = results["log_returns"]

print("简单收益率（前 5 行）:")
display(rets.head())

print("\n手写函数 summary_stats:")
display(results["stats_loop"].round(6))

print("\nNumPy 广播 vectorized_summary（应一致）:")
display(results["stats_broadcast"].round(6))

most_volatile = results["stats_loop"]["std"].idxmax()
print(f"\n波动最大（日收益标准差最高）: {most_volatile}")

In [ ]:
cum_log = results["cumulative_log_return"]
fig, ax = plt.subplots(figsize=(10, 4))
for col in cum_log.columns:
    cum_log[col].plot(ax=ax, label=col)
ax.set_title("累计对数收益曲线")
ax.legend()
plt.tight_layout()
plt.show()

## 第三步：滚动窗口、相关性与时间序列

- 20 / 60 日移动平均线  
- 20 日滚动波动率  
- 收益率相关矩阵  
- `groupby` 月均收益 vs `resample` 月末复合收益

In [ ]:
ma = results["moving_averages"]
display(ma[[c for c in ma.columns if "NVDA" in c]].tail())

print("相关性矩阵:")
display(results["correlation"].round(3))

tech = ["GOOGL", "AVGO", "NVDA"]
tech_corr = results["correlation"].loc[tech, tech].values[np.triu_indices(3, k=1)]
slv_corr = results["correlation"].loc["SLV", tech].mean()
print(f"科技股之间平均相关系数: {tech_corr.mean():.3f}")
print(f"SLV 与科技股平均相关系数: {slv_corr:.3f}")

In [ ]:
print("groupby 月均日收益（最近 6 个月）:")
display(results["monthly_groupby"].tail(6))

print("\nresample 月末复合月收益（最近 6 个月）:")
display(results["monthly_resample"].tail(6))

print("\n滚动波动率异常高点（各标的 99% 分位以上）:")
for ticker, spikes in results["volatility_spikes"].items():
    print(f"  {ticker}: {len(spikes)} 天, 示例日期 {list(spikes.index[:3])}")

## 第四步：图表与结论

图表保存至 `images/` 目录，便于 GitHub 展示。

In [ ]:
paths = generate_all_figures(results)
for p in paths:
    print(p)

### 分析结论（请根据你运行时的最新数字微调）

1. **波动最大：** 通常 NVDA 或 AVGO 的日收益标准差高于 GOOGL；SLV 作为商品 ETF，波动特征与科技股不同，需对照上方 `stats_loop` 表中的 `std` 列确认。
2. **相关性：** 三只科技股（GOOGL、AVGO、NVDA）日收益相关系数普遍为正且较高；SLV 与科技股相关性明显更低，体现股票与大宗商品的不同驱动因素。
3. **滚动波动率：** 2022–2024 期间市场大幅波动阶段，四只标的的 20 日滚动波动率均会出现尖峰；科技股尖峰往往与财报、利率或 AI 主题新闻同步，SLV 则更多反映贵金属价格与宏观预期。
4. **工程结构：** 数据采集在 `src/data_fetch.py`，分析逻辑在 `src/analysis.py`，统计工具在 `src/utils.py`，本 Notebook 负责展示与文字说明。